In [1]:
import pandas as pd
import numpy as np
import math
from statsmodels.tsa.holtwinters import Holt as StatsHolt
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

def simple_exponential_smoothing(data, alpha):
    forecast = [data[0]]
    for i in range(1, len(data)):
        forecast.append(alpha * data[i-1] + (1 - alpha) * forecast[i-1])
    return forecast

def holts_method(data, alpha, beta):
    level = [data[0]]
    trend = [data[1] - data[0]]
    forecast = [data[0]]
    for i in range(1, len(data)):
        level.append(alpha * data[i] + (1 - alpha) * (level[i-1] + trend[i-1]))
        trend.append(beta * (level[i] - level[i-1]) + (1 - beta) * trend[i-1])
        forecast.append(level[i-1] + trend[i-1])
    forecast.append(level[-1] + trend[-1])
    return forecast

def moving_average_forecast(data, window):
    ma = pd.Series(data).rolling(window=window).mean()
    return ma.iloc[-1], ma

def mse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean((y_true - y_pred) ** 2)

def arima_forecast_and_mse(data, product_name=None):
    data = np.array(data, dtype=float)
    if len(data) < 6 or np.count_nonzero(data) < 3 or np.std(data) == 0:
        if product_name:
            print(f"ARIMA skipped for {product_name}: insufficient or constant data.")
        return np.nan, np.nan
    try:
        model = ARIMA(data, order=(1, 1, 1))
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=1)[0]
        fitted = model_fit.fittedvalues
        # Explicitly skip the first fitted value
        y_true = data[2:]
        y_pred = fitted[1:len(y_true)+1]
        mse_val = mse(y_true, y_pred)
        return math.ceil(forecast), mse_val
    except Exception as e:
        if product_name:
            print(f'ARIMA failed for {product_name}: {e}')
        return np.nan, np.nan

In [3]:
df = pd.read_csv('vending_machine_sales.csv')
df['TransDate'] = pd.to_datetime(df['TransDate'])

In [4]:
product_totals = df.groupby('Product')['RQty'].sum().sort_values(ascending=False)
products = product_totals.index.tolist()

In [5]:
prod_df = df[df['Product'] == 'Coca Cola - Zero Sugar'].copy()
monthly = prod_df.groupby(pd.Grouper(key='TransDate', freq='ME'))['RQty'].sum()
# Fill missing months with 0
all_months = pd.date_range(start=monthly.index.min(), end=monthly.index.max(), freq='ME')
monthly = monthly.reindex(all_months, fill_value=0)
data = monthly.values.tolist()

In [6]:
data

[52, 52, 68, 101, 86, 65, 69, 32, 36, 37, 23, 40]

In [7]:
ma3_val, ma3_series = moving_average_forecast(data, 3)
ma5_val, ma5_series = moving_average_forecast(data, 5)

In [9]:
ma3_series

0           NaN
1           NaN
2     57.333333
3     73.666667
4     85.000000
5     84.000000
6     73.333333
7     55.333333
8     45.666667
9     35.000000
10    32.000000
11    33.333333
dtype: float64

In [11]:
ma3_mask = ~ma3_series.isna()

In [13]:
ma3_mask

0     False
1     False
2      True
3      True
4      True
5      True
6      True
7      True
8      True
9      True
10     True
11     True
dtype: bool

In [15]:
mse(data[3:], ma3_series[2:11])

np.float64(561.0246913580246)